# migjorn — lossless MCNP parsing in Python

[`migjorn`](https://github.com/AlvaroCubi/migjorn) is a fast, **lossless** MCNP
input parser written in Rust, with Python bindings.

**Lossless** is the defining property: parse → edit → re-emit reproduces the input
byte-for-byte *except where you changed it*. Comments, spacing, and continuations
all survive.

This notebook is a tour of the whole Python API:

1. [Parse & diagnostics](#parse) — including how malformed input is reported
2. [Reading](#reading) — cells, surfaces, materials, transforms, data cards, parameters
3. [Lookups & counts](#lookups)
4. [Value editing](#values) — materials, densities, coefficients, fractions, displacements
5. [Cell parameters](#params) — `imp:n`, `vol`, `u`, `fill`
6. [Geometry editing](#geometry) — add/remove surfaces and `#n` complements
7. [Building & removing cards](#cards)
8. [Renumbering](#renumber) — the flagship: definitions *and* every reference
9. [Validation](#validate)
10. [Universes: extract & merge](#universes)
11. [Errors](#errors)
12. [Saving](#save)

In [ ]:
import migjorn

print("migjorn version:", migjorn.__version__)

<a id="parse"></a>
## 1. Parse & diagnostics

A small but representative model: line comments, an inline `$` comment, a `&`
continuation, a union (`:`), a cell complement (`#11`), a `LIKE n BUT` cell, an
`RPP` macrobody, a transform-positioned surface, a reflecting (`*`) surface, a
filled universe, and a tally.

In [ ]:
MODEL = """\
Migjorn showcase: a pin cell inside a filled universe
c --- cells ---
1 1 -10.5 -1 imp:n=1 u=10             $ fuel pin (universe 10)
2 2 -1.0 1 -2 imp:n=1 u=10            $ water gap (universe 10)
3 0 2 imp:n=1 u=10                    $ rest of universe 10
10 0 -3 fill=10 imp:n=1               $ shell, filled by universe 10
11 3 -2.7 -40 imp:n=1 imp:p=1 &
     vol=8.0                          $ aluminium insert (macrobody)
12 0 (3 -20) #11 imp:n=1              $ moderator minus the insert
13 0 20 : 50 imp:n=0                  $ graveyard
14 like 11 but mat=4 rho=-11.3        $ second insert, different material

c --- surfaces ---
1 CZ 0.5
2 CZ 0.6
3 CZ 5.0
20 SO 100
30 1 PZ 50                            $ positioned by transform 1
40 RPP -1 1 -1 1 -1 2
*50 PX 60                             $ reflecting boundary

c --- data ---
m1 92235.80c 0.04 92238.80c 0.96      $ enriched uranium
m2 1001.31c 2 8016.31c 1              $ light water
mt2 lwtr.10t
m3 13027.80c 1                        $ aluminium
m4 92235.80c 0.20 92238.80c 0.80
tr1 0 0 5
mode n
sdef pos=0 0 0 erg=2.0
f4:n 1 2 11
"""

print(MODEL)

In [ ]:
model = migjorn.parse(MODEL)   # or: migjorn.Model(MODEL) / migjorn.Model.from_file(path)
model

### Losslessness

Re-emitting reproduces the input exactly — this is the invariant everything else
is built on.

In [ ]:
assert str(model) == MODEL          # str(model) == model.to_source()
assert model.to_source() == MODEL
print("byte-for-byte lossless:", str(model) == MODEL)
print("diagnostics:", model.diagnostics)

### How malformed input is reported

The parser is **recoverable** — it never raises on bad input. Problems surface in
two distinct places:

1. `model.diagnostics` reports whole-file **structure**: the blank lines that
   delimit the cell / surface / data blocks.
2. A **card** that doesn't parse cleanly is *not* a diagnostic — it comes back as
   a typed object with `well_formed == False`, so you spot it while reading.

Either way the text is preserved exactly: losslessness doesn't depend on
understanding the input.

In [ ]:
# 1. A structural problem -> a diagnostic.
no_delim = migjorn.parse("t\n1 0 -1\n\n1 SO 5\nm1 1001 1\n")
for d in no_delim.diagnostics:
    print(f"[{d.severity}] {d.message}  (bytes {d.start}..{d.end})")

# 2. A card-level problem -> well_formed=False, no diagnostic.
bad = migjorn.parse("t\n1 0 (-1\n\n1 CZ 5\n\nm1 1001 1\n")
print("diagnostics:", bad.diagnostics)
print("cell 1 well_formed:", bad.cells[0].well_formed)

# Both still round-trip.
print("still lossless:", str(bad) == "t\n1 0 (-1\n\n1 CZ 5\n\nm1 1001 1\n")

<a id="reading"></a>
## 2. Reading

### Surfaces

Each surface exposes its number, mnemonic, coefficients, an optional transform,
and its boundary condition.

In [ ]:
for s in model.surfaces:
    tr = f"  (TR{s.transform})" if s.transform else ""
    bc = "  [reflecting]" if s.reflective else ("  [white]" if s.white else "")
    print(f"{s.id:>3}  {s.kind:<4} {s.coeffs}{tr}{bc}")

### Cells

Cells expose material/density, void status, the surfaces they reference (with
sense), `#n` complements, and the `LIKE n` base.

In [ ]:
for c in model.cells:
    if c.like is not None:
        print(f"{c.id:>3}  LIKE {c.like} BUT ...")
        continue
    kind = "void" if c.is_void else f"mat {c.material} @ {c.density}"
    extra = f"  complements #{c.cell_refs}" if c.cell_refs else ""
    print(f"{c.id:>3}  {kind:<14} surfaces {c.signed_surfaces}{extra}")

`signed_surfaces` keeps the sense; `surface_ids` gives the bare magnitudes.

In [ ]:
c12 = model.cell(12)
print("signed_surfaces:", c12.signed_surfaces)   # sense preserved
print("surface_ids:    ", c12.surface_ids)       # magnitudes only
print("cell_refs:      ", c12.cell_refs)         # #n complements (+ a LIKE base)
print("well_formed:    ", c12.well_formed)

### Cell parameters, universes & fills

`cell.params` gives every parameter in source order; `cell.param(key)` fetches one
by keyword — bare (`"vol"`) or particle-qualified (`"imp:n"`).

In [ ]:
for c in model.cells:
    bits = []
    for p in c.params:
        star = "*" if p.starred else ""
        name = f"{p.key}:{p.particle}" if p.particle else p.key
        bits.append(f"{star}{name}={p.value}")
    u = f"  u={c.universe}" if c.universe is not None else ""
    f = ""
    if c.fill is not None:
        star = "*" if c.fill.starred else ""
        trs = f" ({c.fill.transform})" if c.fill.transform else ""
        f = f"  {star}fill={c.fill.universe}{trs}"
    print(f"{c.id:>3}  {' '.join(bits)}{u}{f}")

In [ ]:
c11 = model.cell(11)
print("vol   =", c11.param("vol").value)
print("imp:n =", c11.param("imp:n").value)
print("imp:p =", c11.param("imp:p").value)
print("tmp   =", c11.param("tmp"))          # absent -> None
print("universes in the model:", model.universe_ids())

### Materials & transforms

In [ ]:
for m in model.materials:
    comp = "  ".join(f"{zaid}={frac}" for zaid, frac in m.entries)
    print(f"m{m.id:<2} {comp}")

print()
for t in model.transforms:
    print(f"tr{t.id}  displacement={t.displacement}  rotation={t.rotation}  degrees={t.degrees}")

### Data cards (generic)

Every card in the data block, including the ones with no dedicated typed view.

In [ ]:
for d in model.data_cards:
    star = "*" if d.starred else ""
    name = f"{d.name}:{d.particle}" if d.particle else d.name
    print(f"  {star}{name}")

### Card text

`.text` is the card's exact source — inline `$` comments and `&` continuations
included. This is what makes text-based exploration work.

In [ ]:
print(model.cell(11).text)
print("---")
print(repr(model.surface(50).text))

<a id="lookups"></a>
## 3. Lookups & counts

`model.cell(id)` / `.surface(id)` / `.material(id)` / `.transform(id)` use a
cached id index — no need to scan the full list. The `num_*` counts are cheap and
don't build the lists at all.

In [ ]:
print("cells:", model.num_cells, " surfaces:", model.num_surfaces,
      " materials:", model.num_materials, " transforms:", model.num_transforms)
print()
print(model.surface(40))
print(model.cell(12))
print(model.material(1))
print(model.transform(1))
print()
print("missing ids return None:", model.cell(999))

<a id="values"></a>
## 4. Value editing

The typed objects are **live handles** on the model, not snapshots. Assigning to
one writes straight through the lossless engine: only that number moves, and
every other byte of the file stays put.

In [ ]:
demo = migjorn.parse(MODEL)
c1 = demo.cell(1)
print("before:", c1.text.rstrip())

c1.material = 5
c1.density = -10.9
print("after: ", c1.text.rstrip())

Assigning a material to a **void** cell just works — it gains a placeholder
density of `0` to fill in. Assigning material `0` makes it void again and drops
the density.

In [ ]:
c1.material = 0                       # void: the density field disappears
print("void:    ", c1.text.rstrip(), "| is_void =", c1.is_void)
c1.material = 1                       # non-void again, placeholder density 0
print("re-mat:  ", c1.text.rstrip())
c1.density = -10.5
print("density: ", c1.text.rstrip())

### Surfaces, materials & transforms are writable too

In [ ]:
s3 = demo.surface(3)                  # "3 CZ 5.0"
s3.set_coeff(0, 6.0)                  # one coefficient by index
print("surface 3:", s3.text.rstrip())
s3.coeffs = [6.5]                     # or the whole (same-length) list at once
print("surface 3:", s3.text.rstrip())

s3.transform = 1                      # attach TR1
print("with TR1: ", s3.text.rstrip())
s3.transform = None                   # ...and detach it again
print("detached: ", s3.text.rstrip())

In [ ]:
m1 = demo.material(1)                 # enriched uranium
m1.set_fraction(0, 0.05)              # tweak the U-235 atom fraction
m1.set_zaid(1, "92238.70c")           # swap a ZAID's library
print("m1: ", m1.text.rstrip())

t1 = demo.transform(1)                # "tr1 0 0 5"
t1.displacement = (0.0, 0.0, 7.5)
print("tr1:", t1.text.rstrip())
t1.set_rotation([1, 0, 0, 0, 1, 0, 0, 0, 1])   # a rotation matrix, spliced in
print("tr1:", t1.text.rstrip())

In [ ]:
# The inline comment on m1 rode through every one of those edits.
assert "$ enriched uranium" in str(demo)
assert migjorn.parse(str(demo)).diagnostics == []
print("comments preserved and the result re-parses cleanly")

<a id="params"></a>
## 5. Cell parameters

`set_param` rewrites a value **in place** — the keyword never moves, so unlike
remove-then-add the card stays byte-for-byte apart from the new value.

In [ ]:
demo = migjorn.parse(MODEL)
c12 = demo.cell(12)
print("before:  ", c12.text.rstrip())

c12.set_param("imp:n", "4")           # -> True  (matched)
print("missing key ->", c12.set_param("tmp", "1"))   # -> False, no error

c12.add_param("vol=12.5")             # spliced onto the end of the tail
c12.add_param("u=20")
print("adds:    ", c12.text.rstrip())
print("params:  ", [p.key for p in c12.params])
print("universe:", c12.universe)

In [ ]:
print("remove vol ->", c12.remove_param("vol"))
print("remove u   ->", c12.remove_param("u"))
print("again      ->", c12.remove_param("vol"))    # already gone -> False
print("after:   ", c12.text.rstrip())

### Inline comments

`append_comment` splices `$ text` in after the card's last token.

In [ ]:
# On a card with no inline comment, the text is simply appended.
plain = demo.add_cell("30 0 -1 imp:n=1")
plain.append_comment("annotated from Python")
print(plain.text.rstrip())

# Every cell in MODEL already carries a `$` comment. Appending to one of those
# puts the new text *before* the old: MCNP treats everything after the first
# `$` as comment text, so the original survives inside it and the card still
# re-parses cleanly -- but it is no longer a separate comment.
c12.append_comment("second note")
print(c12.text.rstrip())
assert migjorn.parse(str(demo)).diagnostics == []

<a id="geometry"></a>
## 6. Geometry editing

Restructure a cell's region: add or remove surfaces and `#n` complements. Only the
edited card is re-emitted — its parameters and inline comment are preserved, and
every other card stays byte-for-byte.

In [ ]:
demo = migjorn.parse(MODEL)
c2 = demo.cell(2)
print("before:", c2.signed_surfaces, "|", c2.text.rstrip())

c2.add_surface(-3)                    # intersect with -3 (negative = inside)
c2.add_complement(11)                 # AND in  #11
print("remove surface 1 ->", c2.remove_surface(1))   # either sense
print("remove #99       ->", c2.remove_complement(99))  # not there -> False

print("after: ", c2.signed_surfaces, "complements", c2.cell_refs)
print(c2.text.rstrip())

In [ ]:
# The parameters and the comment survived the geometry rewrite.
print("imp:n =", c2.param("imp:n").value, " u =", c2.universe)
assert "$ water gap" in c2.text
print("inline comment kept:", "$ water gap" in c2.text)

<a id="cards"></a>
## 7. Building & removing cards

`add_cell` / `add_surface` / `add_material` take MCNP text, validate it as exactly
one well-formed card, and return a **live handle** you can keep editing.

In [ ]:
demo = migjorn.parse(MODEL)
new_s = demo.add_surface("60 SO 200")
new_m = demo.add_material("m9 6000.80c 1")
new_c = demo.add_cell("21 9 -2.0 -60 imp:n=1")

print("surface: ", new_s.text.rstrip())
print("material:", new_m.text.rstrip())
print("cell:    ", new_c.text.rstrip())

new_c.add_surface(-20)                # editable like any other cell
new_c.append_comment("built from Python")
print("edited:  ", new_c.text.rstrip())
print("validate:", demo.validate())   # [] -- consistent

Removal is by number and returns whether anything matched. It deletes the **card
only** — references are left alone, so `validate()` shows you the fallout.

In [ ]:
print("remove surface 60 ->", demo.remove_surface(60))
print("again             ->", demo.remove_surface(60))   # already gone -> False
print("validate:", demo.validate())                       # now cell 21 dangles

print()
print("remove cell 21    ->", demo.remove_cell(21))
print("remove material 9 ->", demo.remove_material(9))
print("remove transform 1->", demo.remove_transform(1))
print("validate:", demo.validate())   # surface 30 used TR1

<a id="renumber"></a>
## 8. Whole-geometry renumbering

The flagship capability. Renumbering updates **definitions _and_ every reference**
consistently: signed surfaces in geometry, `#n` complements, `LIKE n` bases, cell
material fields, `MT`/`MX` companions, surface transform fields, `fill=`
references, and cell/surface ids inside tally bins.

A mapping is either a **callable** (applied per distinct id) or an explicit
**dict** (unmapped ids are left alone).

In [ ]:
demo = migjorn.parse(MODEL)
print("cell 12 before:", demo.cell(12).text.rstrip())

demo.renumber_surfaces(lambda n: n + 1000)          # callable
demo.renumber_cells({12: 912})                      # dict: only cell 12 moves

print("cell 12 after: ", demo.cell(912).text.rstrip())
print()
print("  surfaces 3,20 -> 1003,1020;  complement #11 unchanged;  id 12 -> 912")

In [ ]:
demo = migjorn.parse(MODEL)
demo.offset_surfaces(1000)              # shorthand for a +delta callable
demo.offset_cells(900)
demo.renumber_materials(lambda n: n + 100)
demo.renumber_transforms(lambda n: n + 5)
demo.renumber_universes(lambda n: n + 50)
demo.renumber_tallies(lambda n: n + 10)

out = str(demo)
checks = {
    "geometry + complement": "912 0 (1003 -1020) #911" in out,
    "LIKE base":             "914 like 911 but" in out,
    "surface definition":    "1040 RPP -1 1 -1 1 -1 2" in out,
    "cell ids in tally bins": "f14:n 901 902 911" in out,
    "reflecting * prefix":   "*1050 PX 60" in out,
    "material definition":   "m101 92235.80c" in out,
    "cell material field":   "901 101 -10.5" in out,
    "MT companion card":     "mt102 lwtr.10t" in out,
    "transform definition":  "tr6 0 0 5" in out,
    "surface transform fld": "1030 6 PZ 50" in out,
    "universe definitions":  "u=60" in out,
    "fill references":       "fill=60" in out,
}
for name, ok in checks.items():
    print(f"  {name:<24} {ok}")
assert all(checks.values())

### ...and everything else is still byte-for-byte

In [ ]:
print("line comment kept:  ", "c --- surfaces ---" in out)
print("inline comment kept:", "$ moderator minus the insert" in out)
print("continuation kept:  ", "imp:p=1 &" in out)
print("odd spacing kept:   ", "     vol=8.0" in out)
print("re-parses cleanly:  ", migjorn.parse(out).diagnostics == [])
print()
print(out)

<a id="validate"></a>
## 9. Validation

`validate()` is a **semantic** check, distinct from parse diagnostics. It reports
duplicate definitions and dangling references.

In [ ]:
print("pristine model:", migjorn.parse(MODEL).validate())

broken = migjorn.parse(MODEL)
broken.remove_surface(40)
broken.remove_material(3)
broken.remove_transform(1)
for problem in broken.validate():
    print(" ", problem)

In [ ]:
# A thrice-defined id is reported exactly once.
dupes = migjorn.parse("t\n1 0 -5\n\n5 SO 1\n5 SO 2\n5 SO 3\n\nm1 1001 1\n")
print(dupes.validate())

<a id="universes"></a>
## 10. Universes: extract & merge

`extract_universe(u)` carves a universe into a standalone model: its cells plus
*everything* they reference — surfaces, materials, transforms, and the cells
reached through `#n`/`LIKE` (followed transitively) — so the result runs on its
own.

In [ ]:
model = migjorn.parse(MODEL)
print("universes:", model.universe_ids())

filler = model.extract_universe(10)
print("universe 10 -> cells", [c.id for c in filler.cells],
      " surfaces", [s.id for s in filler.surfaces],
      " materials", [m.id for m in filler.materials])
print("self-contained:", filler.validate() == [])

shell = model.extract_level0()          # the inverse selection: cells with no u=
print("level 0     -> cells", [c.id for c in shell.cells])

In [ ]:
# Drop the data block for a geometry-only sub-model.
geom = model.extract_universe(10)
geom.clear_data_cards()
print("geometry-only:", len(geom.cells), "cells,", len(geom.data_cards), "data cards")
print(geom)

### Merging disjoint models

In [ ]:
a = migjorn.parse("shell\n1 0 -1 fill=10 imp:n=1\n\n1 SO 5\n\nm1 1001 1\n")
b = migjorn.parse("filler\n2 0 -2 u=10 imp:n=1\n\n2 SO 3\n")
a.merge([b])
print(a)
print("validate:", a.validate())
print()
print(a)

Merging models that share an id is **refused**, with a `MergeError` listing every
collision. A refused merge leaves the model untouched.

In [ ]:
x = migjorn.parse("a\n1 0 -1 imp:n=1\n\n1 SO 5\n\nm1 1001 1\n")
y = migjorn.parse("b\n1 0 -1 imp:n=1\n\n1 SO 9\n\nm1 1001 1\n")

try:
    x.merge([y])
except migjorn.MergeError as e:        # a subclass of ValueError
    print("MergeError:", e)

print()
print("model untouched:", "1 SO 5" in str(x))

<a id="errors"></a>
## 11. Errors

An edit that can't be expressed raises `ValueError` rather than corrupting the
model. A handle whose card was deleted raises `RuntimeError`.

In [ ]:
demo = migjorn.parse(MODEL)

for label, fn in [
    ("density on a void cell",     lambda: setattr(demo.cell(13), "density", -1.0)),
    ("geometry on a LIKE cell",    lambda: demo.cell(14).add_surface(1)),
    ("coefficient out of range",   lambda: demo.surface(3).set_coeff(9, 1.0)),
    ("malformed card text",        lambda: demo.add_cell("this is not a cell")),
]:
    try:
        fn()
        print(f"  {label:<28} (no error?!)")
    except ValueError as e:
        print(f"  {label:<28} ValueError: {e}")

In [ ]:
# Removing a cell's only remaining term would leave it with no geometry.
tiny = migjorn.parse("t\n1 0 -1 imp:n=1\n\n1 SO 5\n\nm1 1001 1\n")
try:
    tiny.cell(1).remove_surface(1)
except ValueError as e:
    print("ValueError:", e)

# A handle to a deleted card is stale.
doomed = tiny.cell(1)
tiny.remove_cell(1)
try:
    doomed.id
except RuntimeError as e:
    print("RuntimeError:", e)

In [ ]:
# None of that changed the model.
print("untouched:", str(demo) == MODEL)

<a id="save"></a>
## 12. Saving

`model.save(path)` writes the re-emitted source; `Model.from_file(path)` reads one
back.

In [ ]:
import tempfile, pathlib

with tempfile.TemporaryDirectory() as tmp:
    path = str(pathlib.Path(tmp) / "renumbered.mcnp")

    m = migjorn.parse(MODEL)
    m.offset_surfaces(1000)
    m.save(path)

    reloaded = migjorn.Model.from_file(path)
    print(reloaded)
    print("round-tripped through disk:", str(reloaded) == str(m))

## That's the whole loop

**parse → inspect → edit → validate → emit**, losslessly, from Python.

Everything is documented inline — try `help(migjorn.Model)`, `help(migjorn.Cell)`,
or `help(migjorn.Surface)`.